# scRNA-seq Preprocessing — PBMC Data

Produces a clean, normalized AnnData object ready to be shared.
Doublets are removed using `scDblFinder_class == "singlet"`.

**Steps:**
1. Load data + calculate QC metrics
2. SoupX ambient RNA correction (optional) + gene filter
3. Doublet scoring (scDblFinder, library-aware)
4. Per-library MAD outlier filtering
5. Doublet removal (scDblFinder_class == "singlet")
6. Normalization (normalize_total + log1p)
7. HVG selection (seurat_v3_paper, ribo/MT excluded)
8. UMAP coloured by QC metrics (PCA-based, for QC inspection only)
9. Save preprocessed object

In [ ]:
# ── paths ──────────────────────────────────────────────────────────────────
DATA_PATH      = "../data_for_practicum.h5ad"
RAW_INPUT_PATH = None   # unfiltered matrix for SoupX; set to None to skip
OUTPUT_DIR     = "/vol/disk/ubuntu/master_practicum_cytokines/data"

# ── library / sample column ────────────────────────────────────────────────
SAMPLE_COL = "library"

# ── derived ────────────────────────────────────────────────────────────────
import os
INPUT_PREFIX = os.path.splitext(os.path.basename(DATA_PATH))[0]
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Input:            {DATA_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
%matplotlib inline

import gc
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc
import seaborn as sns
from rpy2.robjects import numpy2ri, pandas2ri
from rpy2.robjects.conversion import localconverter
from scipy.sparse import csc_matrix
from scipy.stats import median_abs_deviation

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
sc.settings.figdir = OUTPUT_DIR
rcb.logger.setLevel(logging.ERROR)

In [ ]:
def is_outlier_per_group(adata, metric: str, nmads: int, group_col: str):
    """MAD-based outlier detection computed within each library separately."""
    M = adata.obs[metric]
    groups = adata.obs[group_col]
    outlier = pd.Series(False, index=adata.obs_names)
    for grp in groups.unique():
        mask = groups == grp
        vals = M[mask]
        med = np.median(vals)
        mad = median_abs_deviation(vals)
        outlier[mask] = (vals < med - nmads * mad) | (vals > med + nmads * mad)
    return outlier


def is_outlier(adata, metric: str, nmads: int):
    M = adata.obs[metric]
    return (M < np.median(M) - nmads * median_abs_deviation(M)) | (
        np.median(M) + nmads * median_abs_deviation(M) < M
    )


def log_shape(adata, label: str):
    print(f"[{label}]  cells: {adata.n_obs:,}   genes: {adata.n_vars:,}")


def free_memory():
    gc.collect()
    ro.r("gc(verbose = FALSE)")

## Step 1 — Load data + QC metrics

In [ ]:
adata = sc.read_h5ad(DATA_PATH)
adata.var_names_make_unique()
log_shape(adata, "Input")

adata.var["mt"]   = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
adata.var["hb"]   = adata.var_names.str.contains(r"^HB[ABDEGMQZ]\d*(?!\w)")

print(f"MT genes:   {adata.var['mt'].sum()}")
print(f"Ribo genes: {adata.var['ribo'].sum()}")
print(f"HB genes:   {adata.var['hb'].sum()}")

sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True
)
print("QC metrics calculated.")

## Step 2 — Ambient RNA correction (SoupX, optional) + gene filter

In [ ]:
if RAW_INPUT_PATH:
    ro.r("library(SoupX)")

    adata_pp = adata.copy()
    sc.pp.normalize_total(adata_pp, target_sum=1e4)
    sc.pp.log1p(adata_pp)
    sc.pp.pca(adata_pp)
    sc.pp.neighbors(adata_pp)
    sc.tl.leiden(adata_pp, key_added="soupx_groups", flavor="igraph",
                 n_iterations=2, directed=False)
    adata.obs["soupx_groups"] = adata_pp.obs["soupx_groups"]
    del adata_pp

    cells = adata.obs_names
    genes = adata.var_names
    data  = adata.X.T

    print(f"Loading raw data for SoupX from: {RAW_INPUT_PATH}")
    adata_raw = sc.read_h5ad(RAW_INPUT_PATH)
    adata_raw.var_names_make_unique()
    genes_raw = adata_raw.var_names
    cells_raw = adata_raw.obs_names
    data_tod  = adata_raw.X.T
    del adata_raw

    data_csc     = data.tocsc();     data_csc.sort_indices()
    data_tod_csc = data_tod.tocsc(); data_tod_csc.sort_indices()

    x    = data_csc.data.astype(np.float64)
    i    = data_csc.indices.astype(np.int32)
    p    = data_csc.indptr.astype(np.int32)
    dims = np.array(data_csc.shape, dtype=np.int32)

    x_tod    = data_tod_csc.data.astype(np.float64)
    i_tod    = data_tod_csc.indices.astype(np.int32)
    p_tod    = data_tod_csc.indptr.astype(np.int32)
    dims_tod = np.array(data_tod_csc.shape, dtype=np.int32)

    with localconverter(ro.default_converter + pandas2ri.converter + numpy2ri.converter):
        ro.globalenv["x"]            = x
        ro.globalenv["i"]            = i
        ro.globalenv["p"]            = p
        ro.globalenv["dims"]         = dims
        ro.globalenv["x_tod"]        = x_tod
        ro.globalenv["i_tod"]        = i_tod
        ro.globalenv["p_tod"]        = p_tod
        ro.globalenv["dims_tod"]     = dims_tod
        ro.globalenv["genes"]        = np.array(genes)
        ro.globalenv["genes_raw"]    = np.array(genes_raw)
        ro.globalenv["cells"]        = np.array(cells)
        ro.globalenv["cells_raw"]    = np.array(cells_raw)
        ro.globalenv["soupx_groups"] = adata.obs["soupx_groups"].to_numpy()

    ro.r("""
    library(Matrix)
    x <- as.numeric(x);       i <- as.integer(i)
    p <- as.integer(p);       dims <- as.integer(dims)
    x_tod <- as.numeric(x_tod); i_tod <- as.integer(i_tod)
    p_tod <- as.integer(p_tod); dims_tod <- as.integer(dims_tod)

    data     <- new("dgCMatrix", Dim=dims,     x=x,     i=i,     p=p)
    data_tod <- new("dgCMatrix", Dim=dims_tod, x=x_tod, i=i_tod, p=p_tod)
    rownames(data)     <- genes;     colnames(data)     <- cells
    rownames(data_tod) <- genes_raw; colnames(data_tod) <- cells_raw

    sc = SoupChannel(data_tod, data, calcSoupProfile=TRUE)
    sc = setClusters(sc, soupx_groups)
    sc = autoEstCont(sc, doPlot=FALSE)
    out = adjustCounts(sc, roundToInt=TRUE)
    """)

    with localconverter(ro.default_converter + pandas2ri.converter + numpy2ri.converter):
        out_py = ro.conversion.rpy2py(ro.globalenv["out"])

    x_out = np.array(out_py.slots["x"])
    i_out = np.array(out_py.slots["i"])
    p_out = np.array(out_py.slots["p"])
    shape = tuple(out_py.slots["Dim"])
    out_matrix = csc_matrix((x_out, i_out, p_out), shape=shape)

    adata.layers["counts"]       = adata.X.copy()
    adata.layers["soupX_counts"] = out_matrix.T
    adata.X = adata.layers["soupX_counts"]
    log_shape(adata, "After SoupX")
else:
    print("SoupX skipped (RAW_INPUT_PATH is None)")
    adata.layers["counts"] = adata.X

sc.pp.filter_genes(adata, min_cells=20)
log_shape(adata, "After gene filter (min_cells=20)")
free_memory()

## Step 3 — Doublet scoring (scDblFinder, library-aware)

In [ ]:
ro.r("""
library(scDblFinder)
library(SingleCellExperiment)
library(BiocParallel)
""")

data_mat = adata.X.T.tocsc()
data_mat.sort_indices()

x    = data_mat.data.astype(np.float64)
i    = data_mat.indices.astype(np.int32)
p    = data_mat.indptr.astype(np.int32)
dims = np.array(data_mat.shape, dtype=np.int32)

with localconverter(ro.default_converter + numpy2ri.converter):
    ro.globalenv["x"]    = x
    ro.globalenv["i"]    = i
    ro.globalenv["p"]    = p
    ro.globalenv["dims"] = dims

if SAMPLE_COL and SAMPLE_COL in adata.obs.columns:
    n_libs = adata.obs[SAMPLE_COL].nunique()
    print(f"Library-aware doublet detection: '{SAMPLE_COL}' ({n_libs} libraries)")
    with localconverter(ro.default_converter + numpy2ri.converter):
        ro.globalenv["samples_vec"] = adata.obs[SAMPLE_COL].astype(str).to_numpy()
else:
    print("WARNING: running scDblFinder without library awareness")
    ro.r("samples_vec <- NULL")

ro.r("""
x <- as.numeric(x); i <- as.integer(i)
p <- as.integer(p); dims <- as.integer(dims)
data_mat <- new("dgCMatrix", Dim=dims, x=x, i=i, p=p)

set.seed(123)
sce <- scDblFinder(SingleCellExperiment(list(counts=data_mat)), samples=samples_vec)
doublet_score <- sce$scDblFinder.score
doublet_class <- sce$scDblFinder.class
""")

with localconverter(ro.default_converter + pandas2ri.converter + numpy2ri.converter):
    adata.obs["scDblFinder_score"] = ro.conversion.rpy2py(ro.globalenv["doublet_score"])
    adata.obs["scDblFinder_class"] = ro.conversion.rpy2py(ro.globalenv["doublet_class"])

print(adata.obs["scDblFinder_class"].value_counts())
print("Scores stored in adata.obs — no cells removed here. MAD filtering follows.")
free_memory()

## Step 4 — Per-library MAD outlier filtering

In [ ]:
if SAMPLE_COL and SAMPLE_COL in adata.obs.columns:
    n_libs = adata.obs[SAMPLE_COL].nunique()
    print(f"Per-library outlier detection: '{SAMPLE_COL}' ({n_libs} libraries)")
    adata.obs["outlier"] = (
        is_outlier_per_group(adata, "log1p_total_counts", 5, SAMPLE_COL)
        | is_outlier_per_group(adata, "log1p_n_genes_by_counts", 5, SAMPLE_COL)
        | is_outlier_per_group(adata, "pct_counts_in_top_20_genes", 5, SAMPLE_COL)
    )
    adata.obs["mt_outlier"] = (
        is_outlier_per_group(adata, "pct_counts_mt", 3, SAMPLE_COL)
        | (adata.obs["pct_counts_mt"] > 8)
    )
else:
    print("WARNING: falling back to global outlier detection")
    adata.obs["outlier"] = (
        is_outlier(adata, "log1p_total_counts", 5)
        | is_outlier(adata, "log1p_n_genes_by_counts", 5)
        | is_outlier(adata, "pct_counts_in_top_20_genes", 5)
    )
    adata.obs["mt_outlier"] = (
        is_outlier(adata, "pct_counts_mt", 3)
        | (adata.obs["pct_counts_mt"] > 8)
    )

print(f"MAD outliers:    {adata.obs['outlier'].sum():,}")
print(f"MT outliers:     {adata.obs['mt_outlier'].sum():,}")

log_shape(adata, "Before MAD filter")
adata = adata[(~adata.obs.outlier) & (~adata.obs.mt_outlier)].copy()
log_shape(adata, "After MAD + MT filter")

In [ ]:
log_shape(adata, "Before doublet removal")
n_doublets = (adata.obs["scDblFinder_class"] == "doublet").sum()
print(f"Doublets to remove: {n_doublets:,}")
adata = adata[adata.obs["scDblFinder_class"] == "singlet"].copy()
log_shape(adata, "After doublet removal")

## Step 5 — Normalization (normalize_total + log1p)

In [ ]:
scales_counts = sc.pp.normalize_total(adata, target_sum=None, inplace=False)
adata.layers["log1p_norm"] = sc.pp.log1p(scales_counts["X"], copy=True)
del scales_counts
free_memory()
print("log1p_norm layer created")

## Step 6 — HVG selection (seurat_v3_paper, ribo/MT excluded)

In [ ]:
sc.pp.highly_variable_genes(adata, layer="counts", flavor="seurat_v3_paper", n_top_genes=2000, inplace=True)
adata.var.rename(columns={"highly_variable": "hvg"}, inplace=True)
adata.var.drop(columns=["means", "variances", "variances_norm", "highly_variable_rank"],
               errors="ignore", inplace=True)

ribo_in_hvg = adata.var["hvg"] & adata.var_names.str.startswith(("RPL", "RPS"))
mt_in_hvg   = adata.var["hvg"] & adata.var_names.str.startswith("MT-")
excluded     = ribo_in_hvg | mt_in_hvg
adata.var.loc[excluded, "hvg"] = False

print(f"Ribosomal genes excluded from HVGs: {ribo_in_hvg.sum()}")
print(f"Mitochondrial genes excluded:        {mt_in_hvg.sum()}")
print(f"Final HVG count (hvg):               {adata.var['hvg'].sum()}")
print(f"All genes kept in object:            {adata.n_vars:,}")

## Step 8 — Save preprocessed object

## Step 7 — UMAP coloured by QC metrics

In [ ]:
adata.X = adata.layers["log1p_norm"]
adata.var["highly_variable"] = adata.var["hvg"]

sc.pp.pca(adata, svd_solver="arpack", mask_var="highly_variable")
sc.pp.neighbors(adata)
sc.tl.umap(adata)

print("UMAP computed. Plotting QC metrics...")

qc_metrics = [
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
    "pct_counts_ribo",
    "pct_counts_hb",
    "scDblFinder_score",
    "scDblFinder_class",
    SAMPLE_COL,
]
qc_metrics = [m for m in qc_metrics if m in adata.obs.columns]

sc.pl.umap(
    adata,
    color=qc_metrics,
    ncols=3,
    save="_preprocessing_qc.png",
)

In [ ]:
log_shape(adata, "Final preprocessed object")

# Restore X to raw counts before saving
adata.X = adata.layers["counts"]

# Make all obs columns h5ad-writable
for col in adata.obs.columns:
    if adata.obs[col].dtype == object:
        adata.obs[col] = adata.obs[col].astype(str)

out_path = os.path.join(OUTPUT_DIR, f"{INPUT_PREFIX}_preprocessed.h5ad")
adata.write_h5ad(out_path)
print(f"Saved: {out_path}")
print("\nDone. Object contains:")
print(f"  adata.X                        — raw counts")
print(f"  adata.layers['counts']         — raw counts")
print(f"  adata.layers['log1p_norm']     — normalized + log-transformed")
print(f"  adata.var['hvg']               — boolean HVG flag ({adata.var['hvg'].sum()} genes)")
print(f"  adata.obsm['X_pca']            — PCA embedding")
print(f"  adata.obsm['X_umap']           — UMAP embedding (PCA-based, for QC only)")
print(f"  adata.obs['scDblFinder_score'] — doublet score")
print(f"  adata.obs['scDblFinder_class'] — all remaining cells are singlets")